# Market-Basket Analysis on IMDB Top 1000 Movies

**Course:** Security and Privacy in Data Publishing (SPDP Lab — Università degli Studi di Milano)  
**Project:** Market-basket analysis using the IMDB Movies Dataset  
**Dataset:** [IMDB Dataset of Top 1000 Movies and TV Shows](https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows) (Public Domain)  

## Description
Each movie is treated as a **basket**, and the actors listed under `Star1`, `Star2`, `Star3`, `Star4` are treated as **items**.  
We apply the **A-Priori** and **PCY** algorithms to find frequent actor co-occurrences and extract association rules.

---

## 0. Setup & Dataset Download
Upload your Kaggle API token (`kaggle.json`) or manually upload the CSV file.

In [ ]:
# Install dependencies
!pip install kaggle --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import itertools
import time
from collections import defaultdict

matplotlib.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

In [ ]:
# OPTION A: Download via Kaggle API
# Upload your kaggle.json first, then run:
#
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows --unzip

# OPTION B: Manual upload
# from google.colab import files
# uploaded = files.upload()  # upload imdb_top_1000.csv directly

# Load the dataset
df = pd.read_csv('imdb_top_1000.csv')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df[['Series_Title', 'Star1', 'Star2', 'Star3', 'Star4']].head(5)

---
## 1. Data Preprocessing

In [ ]:
def build_baskets(df):
    """
    Convert the dataframe into a list of baskets.
    Each basket = one movie, items = its actors (Star1..Star4).
    Actors are lowercased and stripped to ensure consistent naming.
    """
    actor_cols = ['Star1', 'Star2', 'Star3', 'Star4']
    baskets = []
    for _, row in df.iterrows():
        basket = set()
        for col in actor_cols:
            actor = row[col]
            if pd.notna(actor) and str(actor).strip() != '':
                basket.add(str(actor).strip().lower())
        if len(basket) > 0:
            baskets.append(basket)
    return baskets

baskets = build_baskets(df)
print(f'Total baskets (movies): {len(baskets)}')
print(f'Example basket: {list(baskets[0])}')

# Basket size distribution
sizes = [len(b) for b in baskets]
print(f'Basket size — min: {min(sizes)}, max: {max(sizes)}, avg: {np.mean(sizes):.2f}')

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Actor frequency distribution
from collections import Counter

all_actors = [actor for basket in baskets for actor in basket]
actor_counts = Counter(all_actors)

print(f'Total unique actors: {len(actor_counts)}')
print(f'\nTop 15 most frequent actors:')
for actor, count in actor_counts.most_common(15):
    print(f'  {actor.title():<30} {count} movies')

In [ ]:
# Plot top actors and support threshold analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: top 15 actors
top15 = actor_counts.most_common(15)
names = [a.title() for a, _ in top15]
counts = [c for _, c in top15]
axes[0].barh(names[::-1], counts[::-1], color='#378ADD', edgecolor='none')
axes[0].set_xlabel('Number of movies')
axes[0].set_title('Top 15 most frequent actors')
axes[0].tick_params(labelsize=9)

# Right: how many actors survive different support thresholds
thresholds = range(1, 15)
surviving = [sum(1 for c in actor_counts.values() if c >= t) for t in thresholds]
axes[1].plot(list(thresholds), surviving, marker='o', color='#378ADD', linewidth=2)
axes[1].axvline(x=3, color='#E24B4A', linestyle='--', label='s=3')
axes[1].axvline(x=5, color='#EF9F27', linestyle='--', label='s=5')
axes[1].set_xlabel('Support threshold (s)')
axes[1].set_ylabel('Number of frequent actors')
axes[1].set_title('Frequent actors vs. support threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eda_actors.png', bbox_inches='tight')
plt.show()

---
## 3. A-Priori Algorithm

The A-Priori algorithm exploits **monotonicity**: if an itemset is frequent, all its subsets must also be frequent. This allows us to prune the candidate space significantly.

**Two-pass approach for pairs:**
- Pass 1: count all singleton items → find L₁ (frequent items)
- Pass 2: count only pairs where both items ∈ L₁ → find L₂ (frequent pairs)
- Pass k: generalizes to larger itemsets

In [ ]:
def apriori(baskets, min_support):
    """
    A-Priori Algorithm for finding all frequent itemsets.

    Parameters:
        baskets      : list of sets (each set = one movie's actors)
        min_support  : int, minimum number of baskets an itemset must appear in

    Returns:
        frequent_itemsets : dict mapping frozenset -> support count
        stats             : dict with timing and candidate counts per pass
    """
    frequent_itemsets = {}
    stats = {'pass_times': [], 'candidates': [], 'frequent': []}
    k = 1
    # L_{k-1}: frequent itemsets from previous pass (start with all singletons)
    prev_frequent = None

    while True:
        t0 = time.time()

        # --- Generate candidates Ck ---
        if k == 1:
            # Pass 1: all singletons are candidates
            counts = defaultdict(int)
            for basket in baskets:
                for item in basket:
                    counts[frozenset([item])] += 1

        else:
            # Pass k (k >= 2):
            # Candidate generation: join pairs of (k-1)-frequent itemsets
            # that share a (k-2)-element prefix.
            prev_list = list(prev_frequent.keys())
            counts = defaultdict(int)

            # Build candidate set Ck
            candidates = set()
            for i in range(len(prev_list)):
                for j in range(i + 1, len(prev_list)):
                    union = prev_list[i] | prev_list[j]
                    if len(union) == k:
                        # Pruning: all (k-1)-subsets must be frequent
                        subsets = [frozenset(union - {item}) for item in union]
                        if all(s in prev_frequent for s in subsets):
                            candidates.add(union)

            if not candidates:
                break

            # Count candidates in data
            for basket in baskets:
                for candidate in candidates:
                    if candidate.issubset(basket):
                        counts[candidate] += 1

        # --- Filter: keep only frequent itemsets ---
        lk = {itemset: count for itemset, count in counts.items()
              if count >= min_support}

        elapsed = time.time() - t0
        n_candidates = len(counts)
        n_frequent = len(lk)

        stats['pass_times'].append(elapsed)
        stats['candidates'].append(n_candidates)
        stats['frequent'].append(n_frequent)

        print(f'  Pass {k}: {n_candidates:>6} candidates → {n_frequent:>5} frequent  '
              f'({elapsed:.3f}s)')

        if not lk:
            break

        frequent_itemsets.update(lk)
        prev_frequent = lk
        k += 1

    return frequent_itemsets, stats

In [ ]:
MIN_SUPPORT = 3  # adjust as needed

print(f'Running A-Priori with min_support = {MIN_SUPPORT}...\n')
t_start = time.time()
apriori_itemsets, apriori_stats = apriori(baskets, MIN_SUPPORT)
t_total = time.time() - t_start

print(f'\nTotal frequent itemsets found: {len(apriori_itemsets)}')
print(f'Total time: {t_total:.3f}s')

In [ ]:
# Show top frequent pairs
pairs = {k: v for k, v in apriori_itemsets.items() if len(k) == 2}
print(f'Frequent pairs (support >= {MIN_SUPPORT}): {len(pairs)}\n')
print('Top 10 most frequent actor pairs:')
for itemset, sup in sorted(pairs.items(), key=lambda x: -x[1])[:10]:
    actors = ' & '.join(a.title() for a in itemset)
    print(f'  {actors:<50} support = {sup}')

In [ ]:
# Show frequent triples if any
triples = {k: v for k, v in apriori_itemsets.items() if len(k) == 3}
print(f'Frequent triples (support >= {MIN_SUPPORT}): {len(triples)}')
for itemset, sup in sorted(triples.items(), key=lambda x: -x[1])[:5]:
    actors = ', '.join(a.title() for a in itemset)
    print(f'  {{{actors}}}  support = {sup}')

---
## 4. PCY Algorithm

PCY (Park, Chen, Yu) improves on A-Priori by using the **unused memory in Pass 1** to maintain a hash table of pair counts.  
After Pass 1, the hash table is compressed into a **bitmap**: a bucket is marked 1 (frequent) if its count ≥ s, else 0.  
In Pass 2, a pair is counted **only if** both items are frequent **AND** the pair hashes to a frequent bucket.  
This reduces the number of pairs that need to be counted in Pass 2.

In [ ]:
def pcy(baskets, min_support, num_buckets=1000):
    """
    PCY Algorithm for finding frequent pairs.

    Parameters:
        baskets      : list of sets
        min_support  : int, minimum support threshold
        num_buckets  : int, size of the hash table in Pass 1

    Returns:
        frequent_pairs : dict mapping frozenset -> support count
        stats          : dict with timing and filtering info
    """
    stats = {}

    # ---- PASS 1 ----
    # Count singletons + hash all pairs into bucket table
    t0 = time.time()
    item_counts = defaultdict(int)
    bucket_counts = np.zeros(num_buckets, dtype=np.int32)

    for basket in baskets:
        # Count items
        for item in basket:
            item_counts[item] += 1
        # Hash every pair to a bucket
        for item1, item2 in itertools.combinations(sorted(basket), 2):
            bucket = hash((item1, item2)) % num_buckets
            bucket_counts[bucket] += 1

    # Frequent items L1
    L1 = {item for item, cnt in item_counts.items() if cnt >= min_support}

    # Build bitmap: 1 if bucket count >= min_support
    bitmap = (bucket_counts >= min_support)
    frequent_buckets = int(bitmap.sum())
    stats['pass1_time'] = time.time() - t0
    stats['L1_size'] = len(L1)
    stats['frequent_buckets'] = frequent_buckets
    stats['total_buckets'] = num_buckets

    print(f'  Pass 1: {len(item_counts)} items → {len(L1)} frequent items')
    print(f'          Buckets: {frequent_buckets}/{num_buckets} frequent '
          f'({100*frequent_buckets/num_buckets:.1f}%)  ({stats["pass1_time"]:.3f}s)')

    # ---- PASS 2 ----
    # Count only pairs where: both items in L1 AND pair hashes to frequent bucket
    t0 = time.time()
    pair_counts = defaultdict(int)
    total_candidates = 0
    pcy_filtered = 0

    for basket in baskets:
        freq_items = sorted(basket & L1)
        for item1, item2 in itertools.combinations(freq_items, 2):
            total_candidates += 1
            bucket = hash((item1, item2)) % num_buckets
            if bitmap[bucket]:  # PCY filter
                pair_counts[frozenset([item1, item2])] += 1
            else:
                pcy_filtered += 1

    frequent_pairs = {pair: cnt for pair, cnt in pair_counts.items()
                      if cnt >= min_support}

    stats['pass2_time'] = time.time() - t0
    stats['total_candidates'] = total_candidates
    stats['pcy_filtered'] = pcy_filtered
    stats['frequent_pairs'] = len(frequent_pairs)

    print(f'  Pass 2: {total_candidates} candidate pairs → '
          f'{pcy_filtered} filtered by PCY → '
          f'{len(frequent_pairs)} frequent pairs  ({stats["pass2_time"]:.3f}s)')

    return frequent_pairs, stats

In [ ]:
print(f'Running PCY with min_support = {MIN_SUPPORT}, num_buckets = 1000...\n')
t_start = time.time()
pcy_pairs, pcy_stats = pcy(baskets, MIN_SUPPORT, num_buckets=1000)
t_total_pcy = time.time() - t_start

print(f'\nTotal time: {t_total_pcy:.3f}s')
print(f'Frequent pairs found: {len(pcy_pairs)}')

---
## 5. Comparison: A-Priori vs PCY

In [ ]:
# Verify both algorithms find the same frequent pairs
apriori_pairs = {k for k, v in apriori_itemsets.items() if len(k) == 2}
pcy_pairs_set = set(pcy_pairs.keys())

print('=== Correctness Check ===')
print(f'A-Priori frequent pairs : {len(apriori_pairs)}')
print(f'PCY frequent pairs      : {len(pcy_pairs_set)}')
print(f'Sets are equal          : {apriori_pairs == pcy_pairs_set}')

print('\n=== PCY Filtering Effectiveness ===')
filtered = pcy_stats['pcy_filtered']
total    = pcy_stats['total_candidates']
print(f'Total candidate pairs before PCY filter : {total}')
print(f'Pairs filtered out by PCY bitmap        : {filtered} ({100*filtered/max(total,1):.1f}%)')
print(f'Pairs actually counted in Pass 2        : {total - filtered}')

---
## 6. Association Rules

In [ ]:
def extract_association_rules(frequent_itemsets, min_confidence=0.5):
    """
    Extract association rules from frequent itemsets.

    For each frequent itemset J of size n, we generate n rules:
        J - {j} -> j   for each j in J

    confidence(I -> j) = support(I ∪ {j}) / support(I)
    interest(I -> j)   = confidence(I -> j) - P(j)
    """
    # Total number of baskets for computing P(j)
    n_baskets = len(baskets)

    # Build a support lookup
    support_lookup = frequent_itemsets.copy()

    rules = []
    for itemset, sup_IJ in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for j in itemset:
            I = itemset - frozenset([j])
            sup_I = support_lookup.get(I)
            if sup_I is None:
                continue
            confidence = sup_IJ / sup_I
            # P(j) = support({j}) / n_baskets
            sup_j = support_lookup.get(frozenset([j]), 0)
            p_j = sup_j / n_baskets
            interest = confidence - p_j

            if confidence >= min_confidence:
                rules.append({
                    'antecedent': I,
                    'consequent': frozenset([j]),
                    'support': sup_IJ,
                    'confidence': round(confidence, 4),
                    'interest': round(interest, 4)
                })

    # Sort by confidence descending
    rules.sort(key=lambda x: -x['confidence'])
    return rules


rules = extract_association_rules(apriori_itemsets, min_confidence=0.3)
print(f'Association rules found (confidence >= 0.3): {len(rules)}\n')

print(f'{"Antecedent":<40} {"→ Consequent":<25} {"Conf":>6} {"Interest":>9} {"Sup":>5}')
print('-' * 95)
for r in rules[:15]:
    ant = ', '.join(a.title() for a in r['antecedent'])
    con = ', '.join(a.title() for a in r['consequent'])
    print(f'{ant:<40} → {con:<23} {r["confidence"]:>6.2%} {r["interest"]:>+9.4f} {r["support"]:>5}')

---
## 7. Scalability Analysis

To demonstrate that the solution scales to larger datasets, we run both algorithms on increasing fractions of the data and measure runtime.

In [ ]:
fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
apriori_times = []
pcy_times = []
n_baskets_list = []

for frac in fractions:
    n = max(1, int(len(baskets) * frac))
    sample = baskets[:n]
    n_baskets_list.append(n)

    t0 = time.time()
    apriori(sample, MIN_SUPPORT)
    apriori_times.append(time.time() - t0)

    t0 = time.time()
    pcy(sample, MIN_SUPPORT, num_buckets=1000)
    pcy_times.append(time.time() - t0)

    print(f'  {int(frac*100):3d}% ({n:4d} baskets): '
          f'Apriori {apriori_times[-1]:.3f}s, PCY {pcy_times[-1]:.3f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Runtime vs number of baskets
axes[0].plot(n_baskets_list, apriori_times, marker='o', label='A-Priori',
             color='#378ADD', linewidth=2)
axes[0].plot(n_baskets_list, pcy_times, marker='s', label='PCY',
             color='#EF9F27', linewidth=2, linestyle='--')
axes[0].set_xlabel('Number of baskets')
axes[0].set_ylabel('Runtime (s)')
axes[0].set_title('Runtime vs. dataset size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PCY speedup
speedups = [a/p if p > 0 else 1 for a, p in zip(apriori_times, pcy_times)]
axes[1].bar([f'{int(f*100)}%' for f in fractions], speedups,
            color='#639922', edgecolor='none')
axes[1].axhline(y=1.0, color='#E24B4A', linestyle='--', label='No speedup')
axes[1].set_xlabel('Dataset fraction')
axes[1].set_ylabel('Speedup (Apriori time / PCY time)')
axes[1].set_title('PCY speedup over A-Priori')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('scalability.png', bbox_inches='tight')
plt.show()
print('Scalability plot saved.')

---
## 8. Results Summary

In [ ]:
singletons = {k: v for k, v in apriori_itemsets.items() if len(k) == 1}
pairs      = {k: v for k, v in apriori_itemsets.items() if len(k) == 2}
triples    = {k: v for k, v in apriori_itemsets.items() if len(k) == 3}

print('=' * 55)
print('  RESULTS SUMMARY')
print('=' * 55)
print(f'  Dataset             : IMDB Top 1000 Movies')
print(f'  Total baskets       : {len(baskets)}')
print(f'  Total unique actors : {len(actor_counts)}')
print(f'  Support threshold   : {MIN_SUPPORT}')
print('-' * 55)
print(f'  Frequent singletons : {len(singletons)}')
print(f'  Frequent pairs      : {len(pairs)}')
print(f'  Frequent triples    : {len(triples)}')
print(f'  Association rules   : {len(rules)} (conf ≥ 0.30)')
print('-' * 55)
print(f'  PCY pairs filtered  : {pcy_stats["pcy_filtered"]} '
      f'/ {pcy_stats["total_candidates"]} '
      f'({100*pcy_stats["pcy_filtered"]/max(pcy_stats["total_candidates"],1):.1f}%)')
print('=' * 55)